# Hafta 7 · Kuantum Yazılım Framework'leri ve Donanıma Erişim
**Ders:** Kuantum Hesaplama ve Uygulamaları · **Lab süresi:** ~55 dk · **Ortam:** Google Colab

Bu hafta aynı devreyi (Bell ve GHZ) **dört farklı framework'te** yazıp çalıştırıyoruz: Qiskit, Cirq, PennyLane ve Amazon Braket. Sonuç formatlarını ve **bit sırası** farkını görecek, devreleri **OpenQASM** ile framework'ler arasında taşıyacak, **gürültü modelleri** ile "gerçek donanım gibi" simülasyon yapacak ve bir **sahte (fake) IBM arka ucu** üzerinde transpile edeceğiz.

| Bölüm | Konu | Süre |
|---|---|---|
| 0 | Kurulum (Colab'da 2–4 dk sürebilir) | 4 dk |
| A | Aynı Bell/GHZ devresi 4 framework'te | 10 dk |
| B | Bit sırası: aynı durum, farklı string | 6 dk |
| C | Q# ve CUDA-Q (yalnızca okuma, çalıştırılmaz) | 3 dk |
| D | OpenQASM ile devre taşıma | 6 dk |
| E | Simülatör türleri ve Qiskit primitives | 7 dk |
| F | Gürültü modelleri, Hellinger fidelity, derinlik etkisi | 10 dk |
| G | Okuma (readout) hatası düzeltmesi | 5 dk |
| H | Sahte arka uç: coupling map, SWAP, optimization_level | 7 dk |
| I | Gerçek IBM donanımı (korunan hücre) | okuma |
| J | Alıştırmalar | ödev |

## 0 · Kurulum
Aşağıdaki hücre altı paketi kurar. **Colab'da kurulum birkaç dakika sürebilir**; bir kez çalıştırmanız yeterlidir. Kurulumdan sonra içe aktarma hatası alırsanız *Çalışma zamanı → Oturumu yeniden başlat* yapıp hücreleri tekrar çalıştırın. (`ply`, Cirq'ün OpenQASM okuyucusu için gereklidir.)

In [ ]:
!pip install -q qiskit qiskit-aer pylatexenc cirq-core pennylane amazon-braket-sdk qiskit-ibm-runtime ply

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import time
import numpy as np
import matplotlib.pyplot as plt
from importlib.metadata import version

from qiskit import QuantumCircuit, transpile, qasm2, qasm3
from qiskit.quantum_info import Statevector, SparsePauliOp, hellinger_fidelity
from qiskit.primitives import StatevectorSampler, StatevectorEstimator
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, pauli_error, ReadoutError
import cirq
import pennylane as qml
from braket.circuits import Circuit as BraketCircuit
from braket.devices import LocalSimulator

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
NAVY, BLUE, ORANGE, GRAY = "#1F3A5F", "#2E6DB4", "#D9822B", "#8A94A6"

for p in ["qiskit", "qiskit-aer", "cirq-core", "pennylane", "amazon-braket-sdk", "qiskit-ibm-runtime"]:
    print(f"{p:20s} {version(p)}")

def plot_counts(counts_dict, title="", n=None):
    """{'etiket': {bitstring: sayı}} sözlüklerini yan yana çubuk grafikte çizer."""
    k = len(counts_dict); fig, axs = plt.subplots(1, k, figsize=(3.6*k, 3), sharey=True)
    axs = np.atleast_1d(axs)
    for ax, (name, c) in zip(axs, counts_dict.items()):
        nn = n or len(next(iter(c)))
        keys = [format(i, f"0{nn}b") for i in range(2**nn)]
        tot = sum(c.values())
        ax.bar(keys, [c.get(s, 0)/tot for s in keys], color=BLUE)
        ax.set_title(name, color=NAVY, fontsize=11); ax.tick_params(axis="x", rotation=60)
    axs[0].set_ylabel("oran"); fig.suptitle(title, color=NAVY); plt.tight_layout(); plt.show()
print("hazır")

---
## A · Aynı devre, dört framework
Hedef devre (6. haftadan): **Bell** = `H(q0) → CNOT(q0→q1)`, **GHZ-3** = `H(q0) → CNOT(q0→q1) → CNOT(q1→q2)`. İdeal sonuç: yalnızca `00…0` ve `11…1`, her biri ~%50.

| | Qiskit | Cirq | PennyLane | Braket |
|---|---|---|---|---|
| Kübit | `QuantumCircuit(n)` indeks | `cirq.LineQubit.range(n)` nesne | `wires` (indeks/etiket) | tamsayı indeks, otomatik |
| Kapı | `qc.h(0); qc.cx(0,1)` | `cirq.H(q0), cirq.CNOT(q0,q1)` | `qml.Hadamard(0); qml.CNOT([0,1])` | `Circuit().h(0).cnot(0,1)` |
| Ölçüm | `measure_all()` | `cirq.measure(*q, key="m")` | dönüş değeri: `qml.counts()` | otomatik (hepsi ölçülür) |
| Çalıştır | `StatevectorSampler().run([qc], shots=)` | `cirq.Simulator().run(c, repetitions=)` | QNode çağrısı | `LocalSimulator().run(c, shots=)` |

### A.1 Qiskit

In [ ]:
def bell_qiskit():
    qc = QuantumCircuit(2); qc.h(0); qc.cx(0, 1); qc.measure_all(); return qc

def ghz_qiskit(n=3):
    qc = QuantumCircuit(n); qc.h(0)
    for i in range(n-1): qc.cx(i, i+1)
    qc.measure_all(); return qc

display(ghz_qiskit().draw("mpl"))
sampler = StatevectorSampler(seed=7)
res = sampler.run([bell_qiskit(), ghz_qiskit()], shots=1000).result()
qk_bell = res[0].data.meas.get_counts()     # 'meas' = measure_all'ın oluşturduğu klasik yazmaç
qk_ghz  = res[1].data.meas.get_counts()
print("Qiskit Bell:", qk_bell); print("Qiskit GHZ :", qk_ghz)
print("Ham bit dizileri (ilk 5):", res[1].data.meas.get_bitstrings()[:5])

### A.2 Cirq (Google)
Cirq'te kübitler **nesnedir** (`LineQubit`, `GridQubit`). Ölçümler bir **anahtar** (key) ile saklanır; `histogram` sonuçları **tamsayı** olarak verir.

In [ ]:
def ghz_cirq(n=3):
    q = cirq.LineQubit.range(n)
    ops = [cirq.H(q[0])] + [cirq.CNOT(q[i], q[i+1]) for i in range(n-1)]
    return cirq.Circuit(*ops, cirq.measure(*q, key="m"))

c_bell, c_ghz = ghz_cirq(2), ghz_cirq(3)
print(c_ghz)
sim = cirq.Simulator(seed=7)
cq_bell = sim.run(c_bell, repetitions=1000).histogram(key="m")
cq_ghz  = sim.run(c_ghz, repetitions=1000).histogram(key="m")
print("Cirq Bell (tamsayı anahtar):", cq_bell)
print("Cirq GHZ  (tamsayı anahtar):", cq_ghz)
r = sim.run(c_ghz, repetitions=3); print("Ham ölçüm matrisi (satır = shot, sütun = q0,q1,q2):\n", r.measurements["m"])

### A.3 PennyLane (Xanadu)
PennyLane'de devre bir **Python fonksiyonudur** (QNode). Fonksiyonun **dönüş değeri** ne ölçüleceğini belirler: `qml.counts()`, `qml.probs()`, `qml.expval(...)`, `qml.state()`. Bu yapı 14. haftada PyTorch ile hibrit modellerde çok işimize yarayacak.

In [ ]:
dev = qml.device("default.qubit", wires=3, seed=7)

@qml.set_shots(1000)
@qml.qnode(dev)
def ghz_pl(n=3):
    qml.Hadamard(wires=0)
    for i in range(n-1): qml.CNOT(wires=[i, i+1])
    return qml.counts(wires=range(n))

print(qml.draw(ghz_pl)(3))
pl_bell = {str(k): int(v) for k, v in ghz_pl(2).items()}
pl_ghz  = {str(k): int(v) for k, v in ghz_pl(3).items()}
print("PennyLane Bell:", pl_bell); print("PennyLane GHZ :", pl_ghz)

@qml.qnode(qml.device("default.qubit", wires=2))
def bell_state():
    qml.Hadamard(0); qml.CNOT([0, 1]); return qml.state()
print("PennyLane durum vektörü:", bell_state())

### A.4 Amazon Braket SDK (AWS)
Braket devreleri **zincirleme** (fluent) yazılır. `LocalSimulator` tamamen yerelde çalışır, **AWS hesabı gerekmez**. Aynı kod `AwsDevice(arn)` ile buluttaki gerçek donanıma gönderilebilir (ücretli).

In [ ]:
def ghz_braket(n=3):
    c = BraketCircuit().h(0)
    for i in range(n-1): c.cnot(i, i+1)
    return c

print(ghz_braket(3))
local = LocalSimulator()
br_bell = dict(local.run(ghz_braket(2), shots=1000).result().measurement_counts)
br_ghz  = dict(local.run(ghz_braket(3), shots=1000).result().measurement_counts)
print("Braket Bell:", br_bell); print("Braket GHZ :", br_ghz)

### A.5 Karşılaştırma

In [ ]:
cq_ghz_s = {format(k, "03b"): int(v) for k, v in cq_ghz.items()}   # Cirq tamsayılarını stringe çevir
plot_counts({"Qiskit": qk_ghz, "Cirq": cq_ghz_s, "PennyLane": pl_ghz, "Braket": br_ghz}, "Aynı GHZ devresi, dört framework")

print(f"{'Framework':10s} {'Sonuç tipi':32s} Örnek anahtar")
for name, obj in [("Qiskit", qk_ghz), ("Cirq", cq_ghz), ("PennyLane", ghz_pl(3)), ("Braket", local.run(ghz_braket(3), shots=10).result().measurement_counts)]:
    k0 = next(iter(obj)); print(f"{name:10s} {type(obj).__name__:32s} {k0!r} ({type(k0).__name__})")

---
## B · Bit sırası: aynı durum, farklı string
**Deney:** 3 kübit, yalnızca **q0**'a X kapısı. Fiziksel durum her framework'te aynıdır (q0 = 1, q1 = 0, q2 = 0), ama yazılış farklıdır:

| Framework | Sonuç | Kural |
|---|---|---|
| Qiskit | `'001'` | **little-endian**: q0 en **sağda** (bu dersin kuralı) |
| Cirq | `4` → `'100'` | ölçüm listesindeki **ilk kübit en solda** (en anlamlı bit) |
| PennyLane | `probs` indeks 4 → `'100'` | wire 0 en solda |
| Braket | `'100'` | kübit 0 en solda |

In [ ]:
qc = QuantumCircuit(3); qc.x(0); qc.measure_all()
print("Qiskit   :", StatevectorSampler().run([qc], shots=10).result()[0].data.meas.get_counts())

q = cirq.LineQubit.range(3)
print("Cirq     :", cirq.Simulator().run(cirq.Circuit(cirq.X(q[0]), cirq.measure(*q, key="m")), repetitions=10).histogram(key="m"))

@qml.qnode(qml.device("default.qubit", wires=3))
def x0_pl():
    qml.PauliX(0); return qml.probs(wires=[0, 1, 2])
print("PennyLane:", x0_pl(), "-> dolu indeks:", int(np.argmax(x0_pl())))

print("Braket   :", local.run(BraketCircuit().x(0).i(1).i(2), shots=10).result().measurement_counts)

# Durum vektörü indeksi de farklı:
qsv = QuantumCircuit(3); qsv.x(0)
print("\nQiskit Statevector indeksi    :", int(np.argmax(abs(Statevector(qsv).data))), "(= 0b001)")
print("Cirq final_state_vector indeksi:", int(np.argmax(abs(cirq.Simulator().simulate(cirq.Circuit(cirq.X(q[0]), cirq.I.on_each(q[1:]))).final_state_vector))), "(= 0b100)")

### Dönüştürücü: string'i ters çevir
Big-endian (q0 solda) bir string'i Qiskit sırasına çevirmek için **ters çevirmek** yeterlidir: `s[::-1]`. Cirq'ün tamsayı anahtarlarını önce `n` bitlik stringe çevirip sonra ters çeviririz.

In [ ]:
def to_qiskit_order(counts, n=None, big_endian=True):
    """Herhangi bir framework'ün sayımlarını Qiskit sırasındaki {bitstring: int} sözlüğüne çevirir.
    Anahtarlar tamsayı (Cirq) ya da string (PennyLane/Braket) olabilir."""
    out = {}
    for k, v in counts.items():
        s = format(int(k), f"0{n}b") if not isinstance(k, str) else str(k)
        s = s[::-1] if big_endian else s
        out[s] = out.get(s, 0) + int(v)
    return out

print(to_qiskit_order({4: 10}, n=3))          # Cirq -> {'001': 10}
print(to_qiskit_order({"100": 10}))           # Braket/PennyLane -> {'001': 10}
print(to_qiskit_order({"001": 10}, big_endian=False))   # zaten Qiskit

---
## C · Q# ve CUDA-Q: yalnızca okuma (bu hücreler çalıştırılmaz)
Bu iki framework Colab'da ek kurulum ve (CUDA-Q için) NVIDIA ortamı gerektirdiğinden kodları **yalnızca gösterim amaçlıdır**. Sözdizimi resmi dokümantasyondan alınmıştır.

**Q# (Microsoft Quantum Development Kit, QDK)** — kendi dili olan, tip güvenli bir kuantum programlama dili:
```qsharp
import Std.Diagnostics.*;

operation Main() : (Result, Result) {
    use (q1, q2) = (Qubit(), Qubit());   // kübit ayır (|0⟩ ile başlar)
    H(q1);
    CNOT(q1, q2);
    DumpMachine();                        // durum vektörünü yazdır (simülatörde)
    let (m1, m2) = (M(q1), M(q2));        // ölç: Result = Zero | One
    Reset(q1); Reset(q2);                 // kübitler serbest bırakılmadan önce |0⟩'a döndürülmeli
    return (m1, m2);
}
```

**CUDA-Q (NVIDIA)** — Python/C++ içinde çekirdek (kernel) tanımı; GPU hızlandırmalı simülasyon:
```python
import cudaq

@cudaq.kernel
def ghz(qubit_count: int):
    qvector = cudaq.qvector(qubit_count)
    h(qvector[0])
    for i in range(qubit_count - 1):
        x.ctrl(qvector[i], qvector[i + 1])   # kontrollü X = CNOT
    mz(qvector)

result = cudaq.sample(ghz, 3, shots_count=1000)
print(result)        # { 000:~500 111:~500 }  (bit sırası: ilk ayrılan kübit en solda)
```

---
## D · OpenQASM ile devre taşıma
OpenQASM, devreleri **metin olarak** tanımlayan ortak bir dildir; framework'ler arasında bir **ara temsil (IR)** görevi görür. Yazılım benzetmesi: Java kaynak kodu → **bytecode** → farklı JVM'ler; C → **LLVM IR** → farklı işlemciler.

In [ ]:
qc = QuantumCircuit(3, 3); qc.h(0); qc.cx(0, 1); qc.cx(1, 2); qc.measure([0, 1, 2], [0, 1, 2])
s2 = qasm2.dumps(qc); s3 = qasm3.dumps(qc)
print("----- OpenQASM 2 -----\n" + s2); print("----- OpenQASM 3 -----\n" + s3)

# Geri yükleme (round-trip)
qc_back = qasm2.loads(s2)
print("QASM2 round-trip eşit mi?", qc_back == qc)
print("QASM3 yüklendi:", qasm3.loads(s3).count_ops())

In [ ]:
# Qiskit -> Cirq (OpenQASM 2 üzerinden)
from cirq.contrib.qasm_import import circuit_from_qasm
cc = circuit_from_qasm(s2)
print(cc)
r = cirq.Simulator(seed=1).run(cc, repetitions=1000)
print("Cirq ölçüm anahtarları:", list(r.measurements.keys()))   # c_0, c_1, c_2: her klasik bit ayrı anahtar

# Cirq -> Qiskit
print(cirq.qasm(ghz_cirq(2)))
print(qasm2.loads(cirq.qasm(ghz_cirq(2))))

In [ ]:
# Qiskit -> Braket (OpenQASM 3 üzerinden). Küçük uyarlamalar gerekir:
from braket.ir.openqasm import Program
s3_braket = s3.replace('include "stdgates.inc";', "").replace("cx ", "cnot ")   # Braket'in kapı adı 'cnot'
res = local.run(Program(source=s3_braket), shots=1000).result()
print("Braket (QASM3'ten) :", res.measurement_counts)
print("Braket'in kendi QASM3 çıktısı:\n" + ghz_braket(2).to_ir("OPENQASM").source)

⚠️ **Taşınabilirlik tuzakları:** kapı adları (`cx` ↔ `cnot`), `include` dosyaları, `barrier` gibi yönergeler (Cirq'ün okuyucusu `barrier` tanımaz), ölçüm yazmaçlarının adları ve **bit sırası**. Taşıdıktan sonra her zaman bir **doğrulama testi** (aynı dağılım mı?) yapın.

---
## E · Simülatör türleri ve Qiskit primitives
| Yöntem | Ne saklar? | Bellek | Ne zaman? |
|---|---|---|---|
| Statevector | 2ⁿ genlik | 16·2ⁿ bayt | hata ayıklama, tam genlik, n ≲ 30 |
| Shot tabanlı sampler | durum + örnekleme | SV ile aynı | gerçekçi sayımlar |
| Density matrix | 2ⁿ × 2ⁿ matris | 16·4ⁿ bayt | **gürültü**, n ≲ 12–15 |
| MPS (tensör ağı) | küçük tensörler | dolanıklığa bağlı | çok kübit, az dolanıklık |

In [ ]:
# 1) Statevector: tam genlikler
sv = Statevector(ghz_qiskit(3).remove_final_measurements(inplace=False))
print("GHZ genlikleri:", sv.data.round(3)); print("olasılıklar:", sv.probabilities_dict())

# 2) StatevectorSampler (Qiskit 2.x primitive): sayımlar
print("Sampler:", StatevectorSampler(seed=3).run([ghz_qiskit(3)], shots=500).result()[0].data.meas.get_counts())

# 3) StatevectorEstimator: beklenen değerler (ölçümsüz devre + gözlemlenebilir)
bell_nomeas = QuantumCircuit(2); bell_nomeas.h(0); bell_nomeas.cx(0, 1)
obs = [SparsePauliOp("ZZ"), SparsePauliOp("ZI"), SparsePauliOp("XX")]
ev = StatevectorEstimator().run([(bell_nomeas, obs)]).result()[0].data.evs
print("Bell ⟨ZZ⟩, ⟨ZI⟩, ⟨XX⟩ =", ev.round(3))

In [ ]:
# 4) AerSimulator yöntemleri: süre karşılaştırması
for method, n in [("statevector", 20), ("density_matrix", 10), ("matrix_product_state", 20), ("matrix_product_state", 60)]:
    s = AerSimulator(method=method, seed_simulator=1)
    t0 = time.time(); c = s.run(transpile(ghz_qiskit(n), s), shots=200).result().get_counts()
    print(f"{method:22s} n={n:2d}  süre={time.time()-t0:.2f} sn  farklı sonuç={len(c)}")
print("\nBellek: SV 30 kübit =", 16*2**30/2**30, "GiB;  DM 15 kübit =", 16*4**15/2**30, "GiB")

---
## F · Gürültü: yazılımcı gözüyle hata modelleri
Gerçek donanımda kapılar kusursuz değildir. Bunu, devreye **rastgele araya giren istenmeyen kapılar** olarak modelleriz:

| Hata | Ne olur? | Qiskit Aer |
|---|---|---|
| Bit-flip (p) | p olasılıkla araya X girer | `pauli_error([("X", p), ("I", 1-p)])` |
| Faz hatası (p) | p olasılıkla araya Z girer | `pauli_error([("Z", p), ("I", 1-p)])` |
| Depolarize edici (p) | p olasılıkla durum tamamen rastgeleleşir | `depolarizing_error(p, 1)` |
| Okuma hatası | 0 okunması gereken 1 yazılır (veya tersi) | `ReadoutError([[P(0|0), P(1|0)], [P(0|1), P(1|1)]])` |

In [ ]:
def run_noisy(qc, noise, shots=10000, seed=2):
    s = AerSimulator(noise_model=noise, seed_simulator=seed)
    return s.run(transpile(qc, s, optimization_level=0), shots=shots).result().get_counts()

p = 0.1
for name, err, prep_h in [("bit-flip", pauli_error([("X", p), ("I", 1-p)]), False),
                          ("faz", pauli_error([("Z", p), ("I", 1-p)]), True),
                          ("depolarize", depolarizing_error(p, 1), False)]:
    nm = NoiseModel(); nm.add_all_qubit_quantum_error(err, ["id"])
    qc = QuantumCircuit(1)
    if prep_h: qc.h(0)
    qc.id(0)                       # hata bu "boş" kapıya bağlı
    if prep_h: qc.h(0)             # faz hatasını görmek için X bazında oku
    qc.measure_all()
    c = run_noisy(qc, nm); print(f"{name:11s} P(0) = {c.get('0',0)/10000:.3f}   (beklenen hep 0)")
print("Teori: bit-flip 1-p = 0.9 · faz 1-p = 0.9 · depolarize 1-p/2 = 0.95")

In [ ]:
def my_noise(p1=0.02, p2=0.05, ro=(0.03, 0.05)):
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(p1, 1), ["h", "x", "sx", "rz", "id"])
    nm.add_all_qubit_quantum_error(depolarizing_error(p2, 2), ["cx"])
    if ro: nm.add_all_qubit_readout_error(ReadoutError([[1-ro[0], ro[0]], [ro[1], 1-ro[1]]]))
    return nm

noise = my_noise()
print(noise)
nb = run_noisy(bell_qiskit(), noise, shots=4000, seed=11)
ng = run_noisy(ghz_qiskit(3), noise, shots=4000, seed=11)
plot_counts({"Bell (gürültülü)": nb, "GHZ (gürültülü)": ng}, "Aer NoiseModel ile bozulan histogramlar")

def success_prob(counts, targets):
    return sum(counts.get(t, 0) for t in targets) / sum(counts.values())
print("Bell başarı olasılığı:", success_prob(nb, ["00", "11"]), " Hellinger fidelity:", round(hellinger_fidelity(nb, {"00": .5, "11": .5}), 4))
print("GHZ  başarı olasılığı:", success_prob(ng, ["000", "111"]), " Hellinger fidelity:", round(hellinger_fidelity(ng, {"000": .5, "111": .5}), 4))

**Hellinger fidelity** iki olasılık dağılımının ne kadar benzediğini ölçer (1 = aynı, 0 = hiç örtüşmüyor):
$$F(p, q) = \Big(\sum_x \sqrt{p(x)\, q(x)}\Big)^2$$

### Derinlik arttıkça doğruluk düşer
İdealde **hiçbir şey yapmayan** bir devre kuralım: k kez `X–X` çifti (X·X = I). Her X kapısı p = 0.02 depolarize hata taşısın. ⚠️ `optimization_level=0` kullanıyoruz, yoksa transpiler X–X çiftlerini siler!

In [ ]:
p = 0.02
nmx = NoiseModel(); nmx.add_all_qubit_quantum_error(depolarizing_error(p, 1), ["x"])
ks = [0, 5, 10, 20, 40, 60, 100]; meas = []
for k in ks:
    qc = QuantumCircuit(1)
    for _ in range(k): qc.x(0); qc.x(0)
    qc.measure_all()
    meas.append(run_noisy(qc, nmx, shots=4000, seed=3).get("0", 0) / 4000)
theory = [0.5 + 0.5*(1-p)**(2*k) for k in ks]
for k, m, t in zip(ks, meas, theory): print(f"k={k:3d}  derinlik={2*k:3d}  sim={m:.3f}  teori={t:.3f}")
plt.figure(figsize=(7, 3.3)); plt.plot(ks, theory, "-", color=NAVY, label="teori ½+½(1−p)^(2k)"); plt.plot(ks, meas, "o", color=ORANGE, label="Aer")
plt.axhline(0.5, ls=":", color=GRAY); plt.xlabel("X–X çifti sayısı k"); plt.ylabel("P(0)"); plt.legend(); plt.show()

print("transpile (level=1) sonrası X sayısı:", transpile(qc, optimization_level=1).count_ops().get("x", 0), "  <- hepsi silindi!")

---
## G · Okuma hatası düzeltmesi (kalibrasyon matrisi)
1. Her taban durumu (`00`, `01`, `10`, `11`) **hazırla ve ölç** → okunan dağılımlar kalibrasyon matrisinin **sütunlarıdır**: `A[okunan, gerçek]`.
2. Ölçülen dağılım: **p_ölçülen = A · p_gerçek**.
3. Düzeltme: **p_gerçek ≈ A⁻¹ · p_ölçülen** (`np.linalg.solve`). Negatif çıkan küçük değerleri 0'a kırpıp yeniden normalize ederiz.

In [ ]:
ro_noise = my_noise(0, 0, (0.03, 0.05))       # yalnız okuma hatası
labels = ["00", "01", "10", "11"]
A = np.zeros((4, 4))
for j, lab in enumerate(labels):
    qc = QuantumCircuit(2)
    if lab[1] == "1": qc.x(0)                   # Qiskit sırası: sağdaki karakter q0
    if lab[0] == "1": qc.x(1)
    qc.measure_all()
    c = run_noisy(qc, ro_noise, shots=20000, seed=j)
    A[:, j] = [c.get(l, 0)/20000 for l in labels]
print("Ölçülen kalibrasyon matrisi A[okunan, gerçek]:\n", A.round(4))
A1 = np.array([[0.97, 0.05], [0.03, 0.95]])
print("Teorik A ⊗ A:\n", np.kron(A1, A1).round(4))

In [ ]:
def mitigate(counts, A, labels):
    tot = sum(counts.values())
    p_meas = np.array([counts.get(l, 0)/tot for l in labels])
    p = np.linalg.solve(A, p_meas)
    p = np.clip(p, 0, None); p /= p.sum()
    return dict(zip(labels, p))

raw = run_noisy(bell_qiskit(), ro_noise, shots=20000, seed=5)
fixed = mitigate(raw, A, labels)
print("Ham      :", {l: round(raw.get(l, 0)/20000, 4) for l in labels}, " F =", round(hellinger_fidelity(raw, {"00": .5, "11": .5}), 4))
print("Düzeltmiş:", {l: round(v, 4) for l, v in fixed.items()}, " F =", round(hellinger_fidelity(fixed, {"00": .5, "11": .5}), 4))

---
## H · Sahte (fake) arka uç: gerçek donanımın kısıtları
`qiskit_ibm_runtime.fake_provider` içindeki sahte arka uçlar, emekliye ayrılmış gerçek IBM cihazlarının **bağlantı haritası (coupling map), yerel kapı kümesi ve kalibrasyon verilerini (hata oranları)** içerir. Hesap veya internet gerekmez.

In [ ]:
from qiskit_ibm_runtime.fake_provider import FakeManilaV2
backend = FakeManilaV2()
print("Ad:", backend.name, "| kübit:", backend.num_qubits)
print("Yerel kapılar:", [g for g in backend.operation_names if g in ("cx", "ecr", "cz", "sx", "rz", "x", "measure")])
print("Bağlantılar:", sorted({tuple(sorted(e)) for e in backend.coupling_map.get_edges()}))
print(f"CX(0,1) hatası: {backend.target['cx'][(0,1)].error:.4f}   q0 okuma hatası: {backend.target['measure'][(0,)].error:.4f}")

import networkx as nx
G = nx.Graph(sorted({tuple(sorted(e)) for e in backend.coupling_map.get_edges()}))
plt.figure(figsize=(6, 1.4)); nx.draw(G, {i: (i, 0) for i in G.nodes}, with_labels=True, node_color=NAVY, font_color="white", node_size=500, edge_color=GRAY, width=2); plt.show()

In [ ]:
# Mantıksal "yıldız" GHZ: q0 herkese CNOT. Doğrusal donanımda q0 yalnızca q1'e komşu -> SWAP gerekir!
star = QuantumCircuit(5); star.h(0)
for i in range(1, 5): star.cx(0, i)
star.measure_all()
display(star.draw("mpl"))
print(f"Mantıksal devre: derinlik={star.depth()}, CX={star.count_ops()['cx']}")
for lvl in range(4):
    t = transpile(star, backend, optimization_level=lvl, seed_transpiler=7)
    print(f"optimization_level={lvl}:  derinlik={t.depth():3d}  CX={t.count_ops().get('cx',0):3d}  işlemler={dict(t.count_ops())}")

In [ ]:
# SWAP'i gözle görmek: 3 kübitli doğrusal bağlantı, q0 ile q2 komşu değil
from qiskit.transpiler import CouplingMap
qc = QuantumCircuit(3); qc.h(0); qc.cx(0, 1); qc.cx(0, 2)
t = transpile(qc, coupling_map=CouplingMap.from_line(3), initial_layout=[0, 1, 2], optimization_level=0, seed_transpiler=1)
display(t.draw("mpl"))
print("Eklenen işlemler:", dict(t.count_ops()))
# SWAP'ten sonra mantıksal kübitler farklı fiziksel kablolarda biter; transpiler bunu t.layout ile kaydeder
print("Son yerleşim (mantıksal kübit i -> fiziksel kübit):", t.layout.final_index_layout())

In [ ]:
# Sahte arka ucun gürültü modeliyle simülasyon
noisy_backend = AerSimulator.from_backend(backend, seed_simulator=3)
for n in range(2, 6):
    tq = transpile(ghz_qiskit(n), backend, optimization_level=3, seed_transpiler=7)
    c = noisy_backend.run(tq, shots=4000).result().get_counts()
    print(f"GHZ-{n}: CX={tq.count_ops().get('cx',0)}  Hellinger F = {hellinger_fidelity(c, {'0'*n: .5, '1'*n: .5}):.3f}")

# Aynı işi IBM Runtime'ın SamplerV2'si ile "yerel test modunda" (mode = fake backend) yapmak:
from qiskit_ibm_runtime import SamplerV2
job = SamplerV2(mode=backend).run([transpile(bell_qiskit(), backend, optimization_level=3)], shots=1000)
print("SamplerV2 (fake):", job.result()[0].data.meas.get_counts())

---
## I · Gerçek IBM donanımı (isteğe bağlı)
1. [quantum.cloud.ibm.com](https://quantum.cloud.ibm.com) adresinde ücretsiz hesap açın ve **API anahtarı** oluşturun.
2. **Open Plan** (ücretsiz): 28 günlük kayan pencerede **10 dakika QPU süresi** (Eylül 2026 itibarıyla; koşullar değişebilir, güncel durumu platformdan kontrol edin).
3. Anahtarı **asla** notebook'a yazıp paylaşmayın; Colab'da *Secrets* (🔑) bölümüne `IBM_TOKEN` adıyla ekleyin.

Aşağıdaki hücre `GERCEK_DONANIM = False` iken hiçbir şey yapmaz. İş kuyruğa girer; bekleme süresi dakikalar–saatler olabilir.

In [ ]:
GERCEK_DONANIM = False   # True yaparsanız IBM hesabınızla gerçek QPU'ya iş gönderilir

if GERCEK_DONANIM:
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
    from qiskit.transpiler import generate_preset_pass_manager
    from google.colab import userdata
    service = QiskitRuntimeService(channel="ibm_quantum_platform", token=userdata.get("IBM_TOKEN"))
    real = service.least_busy(operational=True, simulator=False)
    print("Seçilen arka uç:", real.name, real.num_qubits, "kübit")
    pm = generate_preset_pass_manager(backend=real, optimization_level=3)
    isa = pm.run(bell_qiskit())                   # ISA devresi: yalnız donanım kapıları
    job = SamplerV2(mode=real).run([isa], shots=1000)
    print("İş kimliği:", job.job_id())
    counts = job.result()[0].data.meas.get_counts()
    print(counts, "F =", hellinger_fidelity(counts, {"00": .5, "11": .5}))
else:
    print("Gerçek donanım hücresi atlandı (GERCEK_DONANIM = False).")

---
## J · Alıştırmalar
`# TODO` yerlerini doldurun; `assert` satırları geçerse çözüm doğrudur.

### Alıştırma 1 · Cirq histogramını Qiskit sırasına çevir
`cirq_to_qiskit(hist, n)`: Cirq'ün tamsayı anahtarlı histogramını (ilk kübit en anlamlı bit) Qiskit sırasındaki string sözlüğe çevirsin.

In [ ]:
def cirq_to_qiskit(hist, n):
    # TODO
    pass

q = cirq.LineQubit.range(3)
h = cirq.Simulator().run(cirq.Circuit(cirq.X(q[0]), cirq.X(q[1]), cirq.measure(*q, key="m")), repetitions=20).histogram(key="m")
print(h)
assert cirq_to_qiskit(h, 3) == {"011": 20}
assert cirq_to_qiskit({1: 5, 6: 3}, 3) == {"100": 5, "011": 3}
print("Alıştırma 1 ✓")

### Alıştırma 2 · Dört framework, tek sözlük
Braket'te `X(0)` ve `X(2)` uygulanmış 3 kübitli devreyi çalıştırın ve sonucu Qiskit sırasına çevirin. Aynı devreyi Qiskit'te kurup sonuçların **aynı anahtarı** verdiğini doğrulayın. (İpucu: `to_qiskit_order`.)

In [ ]:
# TODO: braket_counts ve qiskit_counts'u üretin
braket_counts = None
qiskit_counts = None

assert set(braket_counts) == set(qiskit_counts) == {"101"}
print("Alıştırma 2 ✓", braket_counts, qiskit_counts)

### Alıştırma 3 · Hellinger fidelity'yi kendin yaz
`my_hellinger(p, q)`: iki sayım/olasılık sözlüğünü normalize edip F = (Σ √(p·q))² hesaplasın. Qiskit'in `hellinger_fidelity` fonksiyonuyla karşılaştırın.

In [ ]:
def my_hellinger(p, q):
    # TODO
    pass

P = {"00": 0.5, "11": 0.5}; Q = {"00": 0.46, "11": 0.44, "01": 0.05, "10": 0.05}
print(my_hellinger(P, Q))
assert np.isclose(my_hellinger(P, Q), hellinger_fidelity(P, Q))
assert np.isclose(my_hellinger(nb, P), hellinger_fidelity(nb, P))
assert np.isclose(my_hellinger(P, P), 1.0)
print("Alıştırma 3 ✓")

### Alıştırma 4 · QASM ile Qiskit → Cirq
4 kübitli GHZ'yi Qiskit'te **ölçümsüz** kurun, `qasm2.dumps` ile metne çevirin, Cirq'e aktarın, Cirq'te tüm kübitleri ölçüp 500 kez çalıştırın. Sonuçlar yalnız `0000` ve `1111` olmalı.

In [ ]:
# TODO: qc4 (Qiskit, ölçümsüz) -> metin -> cirq devresi -> ölç -> sonuç
cirq_counts = None     # {'0000': .., '1111': ..} biçiminde

assert set(cirq_counts) == {"0000", "1111"}
assert sum(cirq_counts.values()) == 500
print("Alıştırma 4 ✓", cirq_counts)

### Alıştırma 5 · Bit-flip modelini doğrula
Yalnızca `x` kapısına p = 0.15 bit-flip hatası ekleyen bir `NoiseModel` kurun. `X` + ölçüm devresini 20000 shot çalıştırıp P(1)'in teorik değer **1 − p = 0.85**'e yakın olduğunu gösterin.

In [ ]:
p = 0.15
nm_bf = None      # TODO
qc = QuantumCircuit(1); qc.x(0); qc.measure_all()
c = run_noisy(qc, nm_bf, shots=20000, seed=9)
p1 = c.get("1", 0) / 20000
print("P(1) =", p1)
assert abs(p1 - (1 - p)) < 0.01
print("Alıştırma 5 ✓")

### Alıştırma 6 · Tek kübitte okuma düzeltmesi (elle)
P(1|0) = 0.04, P(0|1) = 0.10 olan bir kübitte ölçülen dağılım **[0.616, 0.384]** (P(0), P(1)). Kalibrasyon matrisi `A1`'i kurup gerçek dağılımı bulun. Beklenen: [0.6, 0.4].

In [ ]:
A1 = None          # TODO: A[okunan, gerçek]
p_meas = np.array([0.616, 0.384])
p_true = None      # TODO
print(p_true)
assert np.allclose(A1.sum(axis=0), 1)
assert np.allclose(p_true, [0.6, 0.4], atol=1e-9)
print("Alıştırma 6 ✓")

### Alıştırma 7 · Derinlik bütçesi
Her X kapısında depolarize hata p = 0.02 varken X–X zincirinde P(0) = ½ + ½(1 − p)^(2k). **P(0) ≥ 0.75** kalacak en büyük k'yı hesaplayan `max_pairs(p, hedef)` fonksiyonunu yazın ve Aer ile doğrulayın.

In [ ]:
def max_pairs(p, hedef):
    # TODO
    pass

k = max_pairs(0.02, 0.75)
print("k =", k)
assert k == 17
qc = QuantumCircuit(1)
for _ in range(k): qc.x(0); qc.x(0)
qc.measure_all()
pk = run_noisy(qc, nmx, shots=20000, seed=1).get("0", 0) / 20000
print("Aer P(0) =", pk)
assert pk > 0.74
print("Alıştırma 7 ✓")

### Alıştırma 8 · Donanıma uygun yerleşim
`two_qubit_count(qc)` devredeki iki kübitli kapı sayısını versin. Doğrusal GHZ-5 (`ghz_qiskit(5)`: komşu CNOT'lar) ile yıldız GHZ-5'i (`star`) FakeManilaV2'ye `optimization_level=3, seed_transpiler=7` ile transpile edip karşılaştırın. Doğrusal GHZ neden daha ucuz?

In [ ]:
def two_qubit_count(qc):
    # TODO
    pass

lin = transpile(ghz_qiskit(5), backend, optimization_level=3, seed_transpiler=7)
st  = transpile(star, backend, optimization_level=3, seed_transpiler=7)
print("doğrusal:", two_qubit_count(lin), " yıldız:", two_qubit_count(st))
assert two_qubit_count(lin) == 4
assert two_qubit_count(st) > two_qubit_count(lin)
print("Alıştırma 8 ✓")

---
### Haftanın özeti
- Qiskit, Cirq, PennyLane, Braket, Q#, CUDA-Q: **aynı kavramlar**, farklı API ve sonuç formatları
- **Bit sırası:** Qiskit little-endian (q0 sağda); diğerleri q0 solda → `s[::-1]`
- **OpenQASM 2/3** framework'ler arası ortak ara temsil; taşıdıktan sonra mutlaka test edin
- Simülatörler: statevector (tam genlik), sampler (sayım), density matrix (gürültü), MPS (çok kübit, az dolanıklık)
- Gürültü = rastgele araya giren istenmeyen kapılar; derinlik arttıkça doğruluk **üstel** düşer
- Okuma hatası kalibrasyon matrisiyle kısmen **düzeltilebilir**: p ≈ A⁻¹·p_ölçülen
- Gerçek donanım: sınırlı bağlantı → **SWAP** → daha derin, daha gürültülü devre; `optimization_level` önemli

**Gelecek hafta:** Oracle (kara kutu fonksiyon) kavramı ve ilk kuantum algoritmaları: **Deutsch-Jozsa** ve **Bernstein-Vazirani**.